# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan, IIT Madras**  
**Week 10: Application Testing – Concepts, Levels, Test Generation & Pytest**

---

## Table of Contents
1. [Fundamentals of Software Testing](#1-fundamentals-of-software-testing)
2. [Levels of Testing](#2-levels-of-testing)
3. [Test Generation Techniques](#3-test-generation-techniques)
4. [Pytest: Python Testing Framework in Depth](#4-pytest-python-testing-framework)

---

## 1. Fundamentals of Software Testing

### 1.1 Why Test? The Core Purpose
Testing is the process of evaluating a system or its components to determine whether it satisfies the specified requirements and to detect defects. It answers a fundamental question: **Does the software work as intended?** The "intention" is captured in requirements and specifications – formal statements of what the system should do.

Without testing, we have no objective evidence that the application behaves correctly. Even if the code compiles and runs, hidden bugs may corrupt data, produce incorrect results, or crash in edge cases. Testing provides **confidence** that the application meets user needs and can be deployed safely.

**Key questions testing addresses:**
- **Correctness:** Does the output match the expected output for given inputs?
- **Robustness:** Does the system handle invalid or unexpected inputs gracefully, without crashing or corrupting data?
- **Performance:** Does it respond within acceptable time limits under expected load?
- **Usability:** Is the interface intuitive and accessible to the intended audience?
- **Installability:** Can the application be set up in the target environment without errors?

Testing is not a single activity but a continuous mindset. It begins during requirements analysis and continues through development, deployment, and maintenance.

### 1.2 Static vs. Dynamic Testing
**Static testing** involves examining the software’s artifacts (code, design documents, requirements) **without executing** the program.
- **Code reviews / walkthroughs:** A group of developers reads the source code line by line, looking for logical errors, style violations, security flaws, and opportunities for improvement. Because the program isn’t running, this is a form of human analysis.
- **Static analysis tools:** Programs that scan source code for common bug patterns (e.g., uninitialised variables, potential null dereferences, memory leaks). They can enforce coding standards and detect vulnerabilities.
- **Formal verification:** Mathematically proving that a program satisfies its specification using theorem provers or model checkers. This is rigorous but expensive and rarely applied to entire applications.

**Dynamic testing** executes the code with selected inputs and compares the actual output with the expected output. It is the most common form of testing and can be automated. Dynamic testing includes unit tests, integration tests, system tests, and acceptance tests.

The distinction matters because static testing can catch errors early, before the code is even runnable, while dynamic testing validates runtime behaviour.

### 1.3 White‑Box, Black‑Box, and Grey‑Box Testing
These terms describe how much knowledge the tester has of the internal implementation.

**White‑Box Testing (Clear / Glass Box)**
- Tester has full access to the source code and design documents.
- Tests are designed based on the code’s internal logic: control flow (if‑else, loops), data structures, and algorithms.
- **Advantages:** Enables targeted testing of specific code paths, including error‑handling branches that are hard to trigger externally. Can achieve high structural coverage.
- **Disadvantages:** Tests may become tightly coupled to the implementation – a small refactor can break many tests even if the behaviour is unchanged. May overlook missing functionality because the tester only tests what is there, not what should be there. Can give a false sense of security if the tests mirror the code’s assumptions.

**Black‑Box Testing**
- Tester knows only the external interfaces: inputs, outputs, and the specification. No knowledge of internal data structures or algorithms.
- Tests are derived from requirements and use cases.
- **Advantages:** Tests are independent of implementation, so they remain valid even if the code is rewritten. Closer to end‑user perspective. Encourages a clear API design.
- **Disadvantages:** It is harder to achieve high code coverage; complex internal error conditions may remain untested. Debugging failures is more difficult because the tester lacks insight into the code.

**Grey‑Box Testing**
- A blend: the tester has partial knowledge of the internals (e.g., database schema, architectural components, algorithms) but does not examine every line of code.
- Example: Knowing that the system uses a specific sorting algorithm and testing boundary conditions that might stress it.
- Strikes a balance between abstraction and depth; often used in integration and system testing.

### 1.4 Regression Testing
**Regression** occurs when a previously working feature breaks due to a recent change (e.g., a new feature, a bug fix, a library upgrade). **Regression testing** is the practice of re‑running existing tests after every change to ensure that no regressions have been introduced.

A **regression test suite** is a collection of tests that grows over time. Each time a bug is fixed, a new test is added to prevent that specific bug from reappearing. Automation is essential – manual regression testing is impractical for anything beyond trivial projects.

In modern development, regression tests are integrated into **Continuous Integration (CI)** pipelines. Every commit triggers the suite, and if any test fails, the team is alerted immediately. This allows rapid detection and correction of problems.

**Intentional regressions** (breaking changes) are sometimes necessary. In such cases, the old tests must be updated or removed, and consumers of the API must be notified (e.g., through API versioning). Regression suites help identify exactly which tests break, so the impact can be communicated clearly.

### 1.5 Code Coverage Metrics
Coverage metrics measure how thoroughly the test suite exercises the source code. High coverage is a necessary but **not sufficient** condition for a well‑tested system; it indicates which code has been *executed*, not whether the behaviour was *correct*.

**1. Function Coverage**  
Has each function (or method) been called at least once?  
*Example:* If a library contains 100 functions, and the tests only call 80 of them, function coverage is 80%.

**2. Statement (Line) Coverage**  
Has each executable statement been executed?  
*Example:* Given:
```c
int foo(int x, int y) {
    int z = 0;               // statement 1
    if (x > 0 && y > 0) {    // statement 2 (the condition)
        z = x;               // statement 3
    }
    return z;                // statement 4
}
```
A single test `foo(1,1)` executes statements 1, 2 (both parts true), 3, and 4 → **100% statement coverage**.  
Yet `foo(0,1)` and `foo(1,0)` were never tested. Statement coverage is a weak measure.

**3. Branch (Decision) Coverage**  
Has each Boolean expression evaluated both to true and false at least once?  
For the above function, the `if` condition must be taken (true) and skipped (false).  
- `foo(1,1)` → condition true → branch covered (taken).  
- `foo(1,0)` → condition false → branch covered (not taken).  
Now we have 100% branch coverage, but we still haven’t tested the case where `x > 0` is false while `y > 0` is true (`foo(0,1)`) separately; branch coverage doesn’t distinguish *why* a condition became false.

**4. Condition Coverage**  
Has each individual Boolean sub‑expression (condition) evaluated both to true and false?  
In `x > 0 && y > 0`:
- `x > 0` must be tested as true (`foo(1,0)`) and false (`foo(0,1)`).
- `y > 0` must be tested as true (`foo(0,1)`) and false (`foo(1,0)`).
These two tests achieve 100% condition coverage, but neither makes the whole condition true, so the body `z = x` is never executed! Thus 100% condition coverage does not guarantee 100% branch coverage.

**5. Path Coverage**  
Has every possible execution path through the code been taken? This is usually infeasible for all but the simplest programs because loops introduce infinite paths.

**Key Takeaway:** Coverage metrics are useful indicators of untested code, but they must be interpreted carefully. A test suite with 100% coverage can still miss critical logic errors if the tests do not verify the *correctness* of the outcomes.

---

## 2. Levels of Testing

Testing occurs at multiple levels of granularity, from individual functions up to the fully integrated system. Each level serves a distinct purpose.

### 2.1 Requirements Gathering – The Foundation of Testing
Testing begins before any code is written. The **requirements gathering** phase involves identifying stakeholders (users, customers, administrators) and eliciting functional and non‑functional requirements.

Using the **IITM Online Degree dashboard** as an example:
- **Stakeholders:** students (view grades, register courses, download hall tickets), teachers (upload grades, post announcements), administrators (manage user accounts, configure system).
- **Functional requirements:** “A student shall be able to drop a course before the deadline.” “An admin shall be able to add a new course.”
- **Non‑functional requirements:** Page colour, font size, logo placement, response time < 2 seconds.

These requirements, once documented, become the basis for **test cases**. Well‑written requirements avoid ambiguity. For instance, “update course registration” could mean add, drop, or swap courses. Clarifying this with use cases (step‑by‑step scenarios) and examples creates a shared understanding between developers, testers, and stakeholders.

### 2.2 Unit Testing
A **unit** is the smallest testable part of an application – typically a single function, method, or class. **Unit testing** verifies that each unit performs correctly in isolation from the rest of the system.

In an MVC web application, units can be:
- A model method (e.g., `User.set_password()`).
- A controller function (e.g., `list_courses()`).
- A utility function.

**Characteristics of good unit tests:**
- **Fast:** They run in milliseconds; a suite of thousands should complete in seconds.
- **Isolated:** External dependencies (databases, networks, file systems) are replaced with test doubles (mocks, stubs, fakes) so that the test focuses solely on the unit’s logic.
- **Repeatable:** The same result every time; no dependence on random numbers or the current time (unless explicitly controlled).
- **Self‑checking:** The test itself determines pass/fail (usually via assertions); no manual inspection needed.

**Example:** Testing the controller for "student registers for a course."  
- A dummy database is pre‑populated with one student and one course.
- The controller is invoked with a POST request containing valid data.
- The test asserts that the response status is `302` (redirect) and that a new enrollment record exists in the database.
- Additional tests check: what happens if the course is full? If the student is already enrolled? If the course ID is invalid?

**Test‑Driven Development (TDD)** takes this further: write the test *before* writing the implementation code. Initially the test fails (red). Then write the minimum code to pass the test (green). Finally, refactor while keeping the test green. This cycle ensures that every line of production code is tested and that the design is driven by the desired behaviour.

### 2.3 Integration Testing
While unit tests isolate components, **integration testing** verifies that multiple units work together correctly. In a web application, integration tests might exercise the interaction between the controller, the model, and a real (or test) database.

**Levels of integration:**
- **Module integration:** Testing interactions between a few related classes (e.g., Student model + Course model + Enrollment model).
- **Sub‑system integration:** Testing the flow from HTTP request through controller to database and back.
- **System integration:** The entire stack, including external services (e.g., payment gateway, email service).

**Challenges:**
- Setting up the test environment (database, configuration) can be slower and more complex than unit tests.
- Failures may be harder to diagnose because multiple components are involved.

**Continuous Integration (CI)** automates integration testing. Every code commit triggers a build and runs the integration test suite. If any test fails, the team is notified immediately. This prevents “integration hell” – the situation where individual components work perfectly but fail when combined at the last minute. CI tools like GitHub Actions, GitLab CI, and Jenkins are commonly used.

### 2.4 System Testing
**System testing** evaluates the complete, fully integrated application in an environment that mimics production as closely as possible. It goes beyond functional correctness to include non‑functional aspects.

**Key activities:**
- **Functional system testing:** End‑to‑end scenarios performed via the user interface or API. For example, a complete flow: “User logs in → browses available courses → registers for a course → logs out → logs back in and verifies the course appears in their dashboard.”
- **Performance testing:** Measuring response time, throughput, and resource usage under expected and peak loads. Tools like JMeter or Locust simulate many concurrent users.
- **Scalability testing:** Verifying that the application can scale horizontally (adding more servers) or vertically (upgrading hardware) to handle growth.
- **Reliability testing:** Running the system for extended periods to detect memory leaks, resource exhaustion, or gradual performance degradation.
- **Security testing:** Covered in detail later.

System testing often uses **browser automation frameworks** like Selenium, which programmatically control a real browser (Chrome, Firefox) to click buttons, fill forms, and assert page content. This simulates actual user interaction.

Because system tests are slow and require a full deployment environment, they are typically run less frequently than unit tests (e.g., nightly or before a release).

### 2.5 User Acceptance Testing (UAT)
**UAT** is the final validation, performed by the end‑users or stakeholders in a production‑like environment. The goal is to confirm that the system meets their real‑world needs and is ready for deployment.

**Beta testing:** A pre‑release version is given to a limited set of external users. They use the software in their own environment and report bugs or usability issues. Beta testers are more tolerant of occasional glitches and provide valuable feedback that developers might not have anticipated.

**Criteria for UAT success:**
- All critical business scenarios can be completed.
- The system is intuitive and matches user expectations.
- No show‑stopper defects remain.

UAT is often manual; however, automated acceptance tests (using tools like Cucumber/Selenium with behaviour‑driven development) can formalise the acceptance criteria.

---

## 3. Test Generation Techniques

### 3.1 API‑Based Testing (OpenAPI / Swagger)
When an application exposes a RESTful API, the API specification itself can drive test generation. The **OpenAPI Specification (OAS)** describes endpoints, HTTP methods, parameters, request bodies, and response schemas. Because it is machine‑readable, tools can automatically generate test cases.

**Swagger Inspector** and similar tools can:
- Import an OpenAPI definition.
- Auto‑generate requests with valid and invalid data (boundary values, missing required fields, wrong types).
- Execute the requests against a running server and verify that the responses match the schema and status codes.
- Record actual API traffic and replay it as a test suite.

**Benefits:**
- The API spec becomes the **single source of truth** for both documentation, implementation (via code generation), and testing.
- Ensures that the API contract is honoured from all sides.
- Dramatically reduces the manual effort of writing API tests.

### 3.2 Abstract Tests and Model‑Based Testing
**Abstract tests** describe test intent in a semi‑formal, human‑readable way without being tied to a specific implementation language.

*Example abstract test:*
- “Make a GET request to the `/` endpoint and ensure the response contains the text ‘Hello World’.”

This can be translated into an **executable test**:
```python
def test_hello(client):
    rv = client.get('/')
    assert b'Hello World' in rv.data
```
The abstract description is technology‑agnostic; the executable version is for a specific framework (Flask test client). Keeping tests abstract aids portability.

**Model‑Based Testing (MBT)** goes further: it builds a formal model of the system’s behaviour (often as a finite state machine or a decision table) and automatically generates test cases from the model.

*Example:* An authentication flow can be modelled with states:
- `Not Logged In`
- `Login Page Displayed`
- `Logged In`
- `Password Reset`

Transitions between states are triggered by actions (e.g., submit correct credentials → go to `Logged In`; submit wrong credentials → stay at `Login Page` with error). An MBT tool can traverse this model and generate test sequences that cover all states and transitions, including error paths. This ensures systematic coverage of complex workflows that a human might overlook.

### 3.3 GUI Testing and Browser Automation
Testing a web application’s graphical user interface involves verifying visual elements and interactive behaviour. While the semantic structure can be tested via the DOM (presence of specific tags, attributes), **GUI testing** ensures that the visual layout, responsiveness, and client‑side interactions (JavaScript) work correctly.

**Selenium WebDriver** is the industry standard for browser automation. It launches a real browser instance and provides an API to:
- Navigate to URLs.
- Locate elements by ID, CSS selector, XPath, or visible text.
- Simulate clicks, typing, selections.
- Wait for elements to appear or conditions to be met (e.g., AJAX completion).
- Capture screenshots for visual comparison.

*Example:* A Selenium test for login:
```python
driver.get("http://localhost:5000/login")
driver.find_element(By.NAME, "username").send_keys("testuser")
driver.find_element(By.NAME, "password").send_keys("secret")
driver.find_element(By.ID, "submit").click()
assert "Welcome" in driver.page_source
```

**Challenges:** Tests are brittle – small UI changes (e.g., renaming a CSS class) can break tests. They are also slower and require a display server (headless browsers like headless Chrome mitigate this). For applications protected by CAPTCHA, full automation becomes difficult; these parts often require manual testing or stubbed CAPTCHA services in test environments.

### 3.4 Security Testing
Security testing attempts to uncover vulnerabilities that could be exploited to compromise the application’s confidentiality, integrity, or availability.

**SQL Injection testing** (covered in depth earlier) is a prime example:
- Submit malicious strings like `' OR '1'='1` in input fields.
- Test that parameterised queries are used (no raw SQL concatenation).
- Automated tools (e.g., sqlmap) can systematically inject hundreds of payloads.

**Cross‑Site Scripting (XSS)** testing: inject `<script>alert(1)</script>` into fields that are reflected back in HTML; verify that the output is properly escaped.

**Fuzz testing (fuzzing):** Generates a massive volume of random or semi‑random inputs (URLs, form data, file uploads) and feeds them to the application to see if it crashes, hangs, or leaks memory. Fuzzing can uncover buffer overflows, assertion failures, and other robustness issues. For web applications, fuzzers can be pointed at API endpoints and will explore parameter combinations aggressively.

From a development perspective, **white‑box security testing** is essential: the developer knows the database schema, the libraries used, and the architecture, and can specifically craft tests to probe weak points. Black‑box security testing (penetration testing) is also performed, typically by external specialists, to simulate a real attacker.

---

## 4. Pytest: Python Testing Framework in Depth

### 4.1 Pytest Basics and Conventions
**pytest** is a mature, feature‑rich testing framework for Python. It is designed to make tests concise, readable, and easy to write. It is an alternative to the built‑in `unittest` module, and it can run `unittest` tests as well.

**Installation:** `pip install pytest`

**Minimal test:**
```python
# test_sample.py
def func(x):
    return x + 1

def test_answer():
    assert func(3) == 5   # This will fail
```
Run with `pytest` in the terminal. pytest automatically discovers files matching `test_*.py` or `*_test.py`, and within them, functions or methods starting with `test`.

**Output:** For the above, pytest reports:
```
test_sample.py F                                    [100%]
...
>       assert func(3) == 5
E       assert 4 == 5
```
It pinpoints the exact failing assertion and shows the actual vs. expected values.

**Pytest is opinionated:** It relies on naming conventions and plain `assert` statements rather than the `self.assert*` methods of `unittest`. This reduces boilerplate and makes tests read like documentation. Pytest uses introspection to provide detailed failure messages even for complex expressions.

### 4.2 Assertions and Exception Testing
The simple `assert` is the core of pytest. Any Python expression that evaluates to `False` triggers a failure.

**Testing for expected exceptions:**
```python
import pytest

def f():
    raise SystemExit(1)

def test_mytest():
    with pytest.raises(SystemExit):
        f()
```
If the code inside the `with` block does not raise `SystemExit`, the test fails. If a different exception is raised, the test fails. This allows precise verification of error handling.

**Use cases:**
- Verify that accessing an out‑of‑range index raises `IndexError`.
- Verify that a validation function raises `ValueError` on invalid input.
- Ensure that an authenticated endpoint raises `401` when no token is provided.

### 4.3 Test Fixtures
Fixtures are a pytest mechanism for providing a fixed baseline upon which tests can run. They handle **setup** (creating resources) and **teardown** (cleaning up after the test).

**Defining a fixture:**
```python
import pytest

@pytest.fixture
def setup_list():
    return ["apple", "banana"]
```
The `@pytest.fixture` decorator marks a function as a fixture. Its return value is injected into test functions that request it by name.

**Using a fixture:**
```python
def test_apple(setup_list):
    assert "apple" in setup_list

def test_banana(setup_list):
    assert "banana" in setup_list

def test_mango(setup_list):
    assert "mango" in setup_list   # This will fail
```
Each test receives a fresh copy of the list (by default, fixtures are re‑created for each test). Fixtures can be scoped (`function`, `class`, `module`, `session`) to control how often they are rebuilt.

**Teardown with `yield`:**
```python
@pytest.fixture
def temp_db():
    # setup
    db = create_temporary_database()
    yield db
    # teardown: executed after test finishes
    db.close()
    delete_temporary_database()
```
This pattern is essential for tests that need temporary files, directories, or database connections. pytest ensures teardown runs even if the test fails.

### 4.4 Testing Flask Applications with Pytest
Flask provides excellent support for testing via the **test client**. The test client simulates HTTP requests without actually running a server, making tests fast and deterministic.

**Creating an application fixture:**
```python
import pytest
from myapp import create_app, db

@pytest.fixture
def app():
    app = create_app({
        'TESTING': True,
        'SQLALCHEMY_DATABASE_URI': 'sqlite:///:memory:'
    })
    with app.app_context():
        db.create_all()
    yield app
    # teardown: drop tables, close connections
    with app.app_context():
        db.drop_all()

@pytest.fixture
def client(app):
    return app.test_client()
```

**Writing tests with the client:**
- `client.get('/path')` – simulate GET request.
- `client.post('/path', data={...})` – simulate POST request.
- `client.get('/path', follow_redirects=True)` – follow redirects automatically.

**Example:**
```python
def test_login_success(client):
    rv = client.post('/login', data={
        'username': 'testuser',
        'password': 'password'
    }, follow_redirects=True)
    assert b'You were logged in' in rv.data

def test_login_invalid(client):
    rv = client.post('/login', data={
        'username': 'wrong',
        'password': 'wrong'
    }, follow_redirects=True)
    assert b'Invalid username' in rv.data
```

**Testing JSON APIs:**
The test client can also test REST endpoints returning JSON:
```python
def test_get_user(client):
    rv = client.get('/api/user/thejeshgn')
    assert rv.status_code == 200
    json_data = rv.get_json()
    assert json_data['username'] == 'thejeshgn'
```

The test client is a powerful black‑box testing tool for Flask applications, enabling thorough API and view testing without network overhead.

### 4.5 Automated Evaluation with Pytest (Educational Context)
In the course itself, pytest is used for **automated grading**. The instructor provides a set of test files that check student submissions. Example:
```python
# test_public.py
import os

def test_contact_html_exists():
    assert os.path.exists('contact.html'), "contact.html not found"
```
When students submit, the grading system runs `pytest` against their work. Passing all public tests earns some marks; hidden tests validate edge cases. This showcases how pytest can be integrated into a continuous evaluation pipeline.

---

## Summary of Week 10

Testing is an integral, continuous discipline that ensures software meets its requirements and behaves reliably. The key concepts covered:

- **Why test:** To verify correctness, robustness, and usability against specifications.
- **Static vs. dynamic:** Static (review, analysis) vs. dynamic (execution‑based) testing.
- **White‑, black‑, grey‑box:** Refers to the level of knowledge of internal implementation.
- **Regression testing:** Re‑running tests after changes to catch breakages; automated in CI.
- **Coverage metrics:** Statement, branch, condition coverage help quantify test thoroughness but do not guarantee correctness.
- **Levels of testing:** Unit (isolated components), Integration (combined modules), System (full environment), UAT (user validation).
- **Test generation:** Can be API‑driven (OpenAPI), model‑driven, or GUI‑automated (Selenium). Security testing (fuzzing, SQL injection) is essential.
- **Pytest:** A Python testing framework using simple `assert` statements, fixtures for setup/teardown, and excellent Flask integration via the test client.

A robust testing strategy is a combination of all these levels and techniques, automated to provide rapid feedback and confidence in the application’s quality.